# Importing Libraries and Functions

In [39]:
# Import libraries
import kagglehub
import pandas as pd
import numpy as np

import sys
from pathlib import Path

# Import functions from src/features.py
sys.path.append(
    str(Path().resolve().parent)
)

from src.features import (
    get_match_result,
    goal_diff,
    add_position_group,
    squad_rating
)

# Load Data

In [40]:
# Load Cleaned Datasets
results = pd.read_csv("../data/processed/results_clean.csv")
wc2026_draw = pd.read_csv("../data/processed/wc2026_draw_clean.csv")
players = pd.read_csv("../data/processed/players_clean.csv")
elo_ratings = pd.read_csv("../data/processed/elo_clean.csv")

In [41]:
results["date"] = pd.to_datetime(results["date"])
elo_ratings["rank_date"] = pd.to_datetime(elo_ratings["rank_date"])

# Features
## 1. Team Ratings

The average rating of the 26 person squad for each country.

In [42]:
# Generate team ratings and verifying the output
team_ratings = squad_rating(players).sort_values(["squad_size", "squad_rating"], ascending=[False, False])
team_ratings

,country,squad_rating,squad_size,missing_players
0,France,84.230769,26,0
3,England,84.076923,26,0
1,Spain,83.730769,26,0
4,Brazil,83.615385,26,0
8,Germany,83.576923,26,0
6,Argentina,83.038462,26,0
10,Portugal,83.000000,26,0
7,Netherlands,81.730769,26,0
5,Belgium,79.884615,26,0
11,Uruguay,78.269231,26,0


## 2. Elo Features

In [43]:
# Sorting by date for good measure, though it should already be sorted
elo_ratings = elo_ratings.sort_values("rank_date")
results = results.sort_values("date")

# Home team Elo ratings
home_elo = elo_ratings.rename(columns={
    "country_full": "home_team",
    "total_points": "home_elo"
})
# Merge home team Elo ratings with results
results = pd.merge_asof(
    results.sort_values("date"),
    home_elo.sort_values("rank_date"),
    left_on="date",
    right_on="rank_date",
    by="home_team",
    direction="backward"
)

# Away team Elo ratings
away_elo = elo_ratings.rename(columns={
    "country_full": "away_team",
    "total_points": "away_elo"
})

# Merge away team Elo ratings with results
results = pd.merge_asof(
    results.sort_values("date"),
    away_elo.sort_values("rank_date"),
    left_on="date",
    right_on="rank_date",
    by="away_team",
    direction="backward"
)

1. elo_diff feature

The difference in elo rating between home and away teams at the time of each match.

In [44]:
results["elo_diff"] = results["home_elo"] - results["away_elo"]

2. home_higher_elo feature

If the home_team has a higher elo rating than the away team, the value is 1. Else, 0.

In [45]:
results["home_higher_elo"] = (results["home_elo"] > results["away_elo"]).astype(int)

In [46]:
results[["date", "home_team", "away_team", "home_elo", "away_elo", "elo_diff", "home_higher_elo"]].sample(10)

,date,home_team,away_team,home_elo,away_elo,elo_diff,home_higher_elo
47577,2024-09-09,Puerto Rico,Aruba,NaN,NaN,NaN,0
33787,2010-05-27,South Africa,Colombia,392.0,776.0,-384.0,0
7147,1967-10-01,Russia,Switzerland,NaN,NaN,NaN,0
37108,2013-07-25,Japan,Australia,689.0,671.0,18.0,1
48836,2025-11-13,Armenia,Hungary,NaN,NaN,NaN,0
20756,1995-12-16,Thailand,Vietnam,NaN,NaN,NaN,0
4167,1955-11-06,Netherlands,Norway,NaN,NaN,NaN,0
42488,2019-06-07,Georgia,Gibraltar,NaN,NaN,NaN,0
40768,2017-06-28,Seychelles,Mozambique,NaN,NaN,NaN,0
38575,2015-03-30,Qatar,Slovenia,300.0,NaN,NaN,0


In [49]:
with pd.option_context('display.max_seq_items', None):
    print(results["away_team"].unique())

<StringArray>
[                         'England',                         'Scotland',
                            'Wales',                 'Northern Ireland',
                           'Canada',                        'Argentina',
                          'Hungary',                   'Czechoslovakia',
                          'Austria',                          'Uruguay',
                           'France',                      'Switzerland',
                         'Alderney',                         'Guernsey',
                      'Netherlands',                          'Belgium',
              'Trinidad and Tobago',                           'Jersey',
                          'Germany',                           'Norway',
                          'Denmark',                           'Sweden',
                            'Italy',                            'Chile',
                        'Catalonia',                          'Finland',
                           'Russia', 